In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from netCDF4 import Dataset

from scintill_ai.io import get_magnetometer_data, get_solar_data, get_solar_wind_data
from scintill_ai.io_async import get_aggregated_gnss_data_by_month_async
from scintill_ai.preprocess import get_solar_position
from var import START_DATE, END_DATE, DATA_IN, DATA_OUT, SMAG_YEARS, SMAG_STATIONS

## INTERMAGNET 🚫

In [ ]:
# df_kou = get_magnetometer_data(
#     Path(DATA_IN, 'KOU')
# ).loc[START_DATE:END_DATE, 'h']

# df_ttb = get_magnetometer_data(
#     Path(DATA_IN, 'TTB')
# ).loc[START_DATE:END_DATE, 'h']

In [ ]:
# df_mag = pd.merge(
#     left=df_kou,
#     right=df_ttb,
#     how='inner',
#     left_index=True,
#     right_index=True,
#     suffixes=['_kou', '_ttb']
# )

## SuperMAG

In [ ]:
dfs_smag = {}

for yr_ in SMAG_YEARS:
    filepath = Path(DATA_IN, 'supermag', f'all_stations_all{yr_}.netcdf')
    dfs_smag[yr_] = get_magnetometer_data(file_path=filepath, stations_list=SMAG_STATIONS)

df_smag = pd.concat(dfs_smag.values(), axis=0)
df_smag['h_tmk'] = df_smag['h_ttb'] - df_smag['h_kou']

## GFZ

In [ ]:
df_solar = get_solar_data(START_DATE, END_DATE)

## OMNIweb

In [ ]:
df_omni = get_solar_wind_data(Path(DATA_IN, 'omniweb')).loc[
    START_DATE:END_DATE,
    ['field_magnitude_avg', 'wind_speed', 'wind_density', 'wind_pressure', 'eletric_field']
]

## ISMR

$$ S_{4,\hspace{0.15 em}\mathrm{denoised}} = \mathrm{Re}\left( \sqrt{S_4^2 - S_{4,\hspace{0.15 em}\mathrm{noise}}^2} \right) $$

In [ ]:
station_name = "PRU4"
output_dir = Path(DATA_OUT, 'ismr', f'{station_name.lower()}')

To download data, uncomment the code below:

In [ ]:
start = "2023-11-01"
end = "2023-12-31"
fields = "time_utc, svid, azim, elev, s4, s4_correction" #, locktime_l1"

df = await get_aggregated_gnss_data_by_month_async(start, end, station_name, fields, output_dir)

In [ ]:
dfs_ismr = dict()

for file_ in output_dir.iterdir():
    dfs_ismr[file_] = pd.read_pickle(file_).set_index('time_utc')

df_ismr = pd.concat(objs=dfs_ismr.values())
df_ismr.index = pd.to_datetime(df_ismr.index.rename('datetime'))
df_ismr = df_ismr.reindex(
    pd.date_range(start=START_DATE, end='2025-01-01', inclusive='left', freq='min'),
)
df_ismr['n_sat'] = np.where(
    df_ismr['n_sat'].isna(), 0, df_ismr['n_sat'],
)

## Full dataset

In [ ]:
df = df_ismr[
    ['n_sat', 's4_max', 's4_mean', 'perc_mild_scint', 'perc_strong_scint']
].merge(
    df_omni,
    how='outer',
    left_index=True,
    right_index=True,
).merge(
    df_smag['h_tmk'],
    how='outer',
    left_index=True,
    right_index=True,
).merge(
    df_solar['f10.7_adj'],
    how='outer',
    left_index=True,
    right_index=True,
)

df['sza'] = get_solar_position(
    df.index, columns=['zenith'], altitude=0,
).round(1)
df['f10.7_adj'] = df['f10.7_adj'].ffill()

In [ ]:
df.to_pickle(Path(DATA_OUT, 'df.pickle'))